# Reddit Kaggle Competition: Adrian Barrios

Multi-label emotion classification over 28 categories. Metric: **Macro ROC AUC**.

### Architecture
BiLSTM (2 layers, bidirectional) + a **Linear Attention** layer that learns a weighted sum over the LSTM outputs, followed by a linear classifier.

### Training
- Data augmentation: random word deletion and adjacent word swap (only on the training split).
- Adam + `ReduceLROnPlateau` on val ROC-AUC, gradient clipping.
- The full training run is executed on a Modal GPU via `modal_train.py` (15 epochs). This notebook reproduces the architecture locally for quick verification.

In [ ]:
import json
import os
import random
import re
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

DATA_DIR = '../data/'
BATCH_SIZE = 128
MAX_LEN = 50
EPOCHS = 3  # short local run; the production model is trained for 15 epochs on Modal
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"Using device: {DEVICE}")

## 1. Load the Data

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_kaggle.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_kaggle.csv'))

emotion_cols = [c for c in train_df.columns if c not in ['ID', 'text']]
num_labels = len(emotion_cols)

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Total emotions: {num_labels}")

## 2. Tokenization & Vocabulary
Lowercase word-level tokens. Vocab built from the 20k most frequent training tokens; `<PAD>=0`, `<UNK>=1`.

In [ ]:
def tokenize(text):
    return re.findall(r'\b\w+\b', str(text).lower())

counts = Counter(t for txt in train_df['text'] for t in tokenize(txt))
vocab = {w: i + 2 for i, (w, _) in enumerate(counts.most_common(20000))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1
vocab_size = len(vocab)

def encode(tokens):
    ids = [vocab.get(w, 1) for w in tokens][:MAX_LEN]
    if not ids:
        ids = [1]  # <UNK> fallback so the attention mask has at least one valid position
    return ids + [0] * (MAX_LEN - len(ids))

with open('vocab.json', 'w') as f:
    json.dump(vocab, f)

print(f"Vocabulary size: {vocab_size}")

## 3. Data Augmentation
Two cheap token-level perturbations applied on the training split only:
- **Random deletion** of tokens with `p_drop=0.1`.
- **Adjacent swap** with `p_swap=0.1`.

Together these make the model more robust to surface variations without changing label semantics.

In [ ]:
def augment(tokens, p_drop=0.1, p_swap=0.1):
    if len(tokens) > 1:
        kept = [t for t in tokens if random.random() > p_drop]
        if kept:
            tokens = kept
    if len(tokens) > 1 and random.random() < p_swap:
        i = random.randint(0, len(tokens) - 2)
        tokens[i], tokens[i + 1] = tokens[i + 1], tokens[i]
    return tokens

## 4. Dataset and DataLoader

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, df, is_test=False, do_augment=False):
        self.tokens = [tokenize(t) for t in df['text'].tolist()]
        self.is_test = is_test
        self.do_augment = do_augment
        if not is_test:
            self.labels = df[emotion_cols].values.astype(np.float32)

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, idx):
        toks = list(self.tokens[idx])
        if self.do_augment:
            toks = augment(toks)
        x = torch.tensor(encode(toks), dtype=torch.long)
        if self.is_test:
            return x
        return x, torch.tensor(self.labels[idx])

train_split, val_split = train_test_split(train_df, test_size=0.1, random_state=SEED)
train_loader = DataLoader(EmotionDataset(train_split, do_augment=True), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(EmotionDataset(val_split), batch_size=BATCH_SIZE)
test_loader = DataLoader(EmotionDataset(test_df, is_test=True), batch_size=BATCH_SIZE)

## 5. Model: BiLSTM + Linear Attention
`LinearAttention` scores each timestep with a `Linear(hidden, 1)` head, softmaxes the scores (with padding masked out), and returns the weighted sum of LSTM outputs as a fixed-size sentence representation. This replaces the final-hidden-state pooling used in the baseline.

In [ ]:
class LinearAttention(nn.Module):
    """Additive attention with a learnable linear scoring layer."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_out, mask):
        # Fall back to position 0 for any all-padding row so softmax never sees
        # an all -inf row (which would produce NaN).
        safe_mask = mask.clone()
        safe_mask[:, 0] |= ~mask.any(dim=1)
        scores = self.attn(lstm_out).squeeze(-1)
        scores = scores.masked_fill(~safe_mask, float('-inf'))
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (weights * lstm_out).sum(dim=1)

class BiLSTMAttentionModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=200, hidden_dim=256, output_dim=28,
                 dropout_p=0.4, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout_p)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout_p if num_layers > 1 else 0.0,
        )
        self.attention = LinearAttention(hidden_dim * 2)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        mask = x != 0
        embedded = self.dropout(self.embedding(x))
        lstm_out, _ = self.lstm(embedded)
        context = self.attention(lstm_out, mask)
        return self.fc(self.dropout(context))

model = BiLSTMAttentionModel(vocab_size, output_dim=num_labels).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)
criterion = nn.BCEWithLogitsLoss()

## 6. Training Loop
Tracks both **train** and **val** ROC-AUC so we can detect overfitting. The full 15-epoch run is executed on Modal (`python -m modal run modal_train.py`); the loop below is the same logic and lets you sanity-check locally.

In [ ]:
def evaluate(loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in loader:
            logits = model(x.to(DEVICE))
            preds.append(torch.sigmoid(logits).cpu().numpy())
            targets.append(y.numpy())
    return roc_auc_score(np.vstack(targets), np.vstack(preds), average='macro')

best_auc = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    val_auc = evaluate(val_loader)
    scheduler.step(val_auc)
    print(f"Epoch {epoch} | Val ROC AUC: {val_auc:.4f}")

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), 'model_best.pth')
        print(f"  ↑ new best, saved to model_best.pth")

print(f"\nBest val ROC-AUC: {best_auc:.4f}")

## 7. Generate Submission (local run)
For the final submission we load the Modal-trained `model_best.pth`; this cell just verifies the local checkpoint runs end-to-end.

In [ ]:
model.load_state_dict(torch.load('model_best.pth', map_location=DEVICE))
model.eval()
probs = []
with torch.no_grad():
    for x in test_loader:
        probs.append(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy())

submission = pd.DataFrame(np.vstack(probs), columns=emotion_cols)
submission.insert(0, 'ID', test_df['ID'].values)
submission.to_csv('submission.csv', index=False)
print('Submission file created: submission.csv')